# D2-S05: Planning scenarios for the CFO

**Which planning case should the CFO use for next quarter?**

Finance expects the **direct costs it buys in** (delivery and subcontractors) to **rise about 12%**, and **sales to
grow about 5%**, above the baseline level for January to March 2026. Overhead is unchanged. The CFO wants that case compared with the original growth plan and with a **less severe case
of your own**, lying between finance's case and the original growth plan. You will check the starting forecast,
choose the assumptions, check one month by hand, and recommend how the CFO should use the comparison.

**How this notebook works**
- Work from top to bottom. Click a code cell and press **Shift+Enter**.
- Code cells are labelled **SUPPLIED** (run it) or **YOUR CHOICE** (change the numbers, then run it).
- Checks print **OK**, **PROBLEM** or **NOT DONE YET**.
- The kernel (top right) must be the course `.virtual-env-folder` environment.

**Rules for this task**
- Fictional Northbridge data in EUR. Revenue is positive and costs are negative; operating result is the sum.
- The forecast may only use information up to **December 2025**.
- A **multiplier** scales a baseline amount once: 1.00 means unchanged, 1.05 means 5% higher. It is applied to each
  month's own baseline, not compounded month after month.
- Overhead stays at 1.00 in both of your cases.
- Scenarios are assumptions. They have no probabilities attached.

## Step 0: Set up
**Do:** run the cell. **You should see:** `Course folder found:` followed by your folder.

In [ ]:
# SUPPLIED: course setup. It finds the course folder so the file paths below work wherever the
# course folder is saved. You do not need to read or change this cell.
import os, sys
from pathlib import Path
COURSE_FOLDER = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case_pack").is_dir()), None)
if COURSE_FOLDER is None:
    raise SystemExit("Open the course folder in VS Code (File > Open Folder), then run this cell again.")
os.chdir(COURSE_FOLDER)
sys.path.insert(0, str(COURSE_FOLDER))
print("Course folder found:", COURSE_FOLDER)

## Step 1: Load the three planning tables (5 minutes)
**Why:** each table has one job. Knowing what one row means stops you mixing observed figures with assumptions.

| Table | File (in `case_pack/data/clean/`) | One row means |
|---|---|---|
| `history` | `history/monthly_actuals_history.csv` | What one unit actually recorded on one report line in one month, 2023 to 2025 |
| `baseline` | `planning/baseline_forecast.csv` | The starting forecast for one unit, report line and month, January to March 2026 |
| `assumptions` | `scenario_assumptions.csv` | One original scenario and its three multipliers |

**Do:** run the cell. **You should see:** 324, 27 and 3 rows, then the first rows of each table.

In [ ]:
# SUPPLIED: read the three tables and show their first rows.
from case_pack.course_tools import planning
history, baseline, assumptions = planning.load_inputs()
display(history.head(3), baseline.head(3), assumptions)

## Step 2: Check the baseline before relying on it (10 minutes)
**Why:** every scenario is built on the baseline, so a wrong baseline makes every scenario wrong.

The baseline uses **seasonal naive**: each future month copies the same month one year earlier (February 2026 copies
February 2025). "Naive" is the method's name: it assumes nothing changes except the time of year.

**Do:** run the cell. **You should see:** three OK lines and a chart of three years of history.

**Discuss with your neighbour:** do you see a pattern that repeats each year, a level that rises or falls over time,
or both? What would copying last year miss if the level is rising?

In [ ]:
# SUPPLIED: check the baseline and show the history chart.
planning.check_baseline(history, baseline)
planning.history_chart(history)

## Step 3: How accurate has this method been before? (10 minutes)
**Why:** you cannot test a forecast of the future, but you can pretend it is September 2025, forecast October to
December from the previous year, and compare with what actually happened. The withheld months are called a
**holdout**.

The table shows the **average size of the error** for each report line: each miss counted as a positive amount,
then averaged (errors of -10 and +20 give an average size of 15). This is also called the mean absolute error (MAE).

**Do:** run the cell. Keep these three numbers in mind: you will use them in Step 7.

In [ ]:
# SUPPLIED: forecast October to December 2025 from 2024 and measure the errors.
planning.holdout_errors(history)

## Step 4: See how a scenario changes one number (10 minutes)
**Why:** before choosing multipliers, be sure what one does to a negative cost.

The original **growth** scenario multiplies revenue by 1.10 and direct cost by 1.08. Take N02's February 2026 direct
cost baseline, -19,648.94 EUR. Under a 1.20 multiplier (not one of today's cases, just to see the arithmetic):

`-19,648.94 x 1.20 = -23,578.728`, rounded to **-23,578.73 EUR**.

A cost multiplied by more than one becomes **more negative**, so the operating result falls.

**Do:** run the cell. It applies the three original scenarios. **You should see:** two OK lines.

In [ ]:
# SUPPLIED: apply the original scenarios (baseline, growth, cost_pressure) and check them.
original = planning.original_scenarios(baseline, assumptions)

## Step 5 (YOUR CHOICE): Set finance's case and your own case (20 minutes)
**Why:** the calculation is supplied; choosing and justifying the assumptions is the planning work.

1. **Finance's case:** translate "sales about 5% above baseline" and "direct costs about 12% above baseline" into
   two multipliers. Northbridge already has a `cost_pressure` scenario with the same 1.12 on direct cost, but no
   revenue growth and 5% on overhead, so finance's expectation is a different case and you build it separately.
2. **Your case:** choose a revenue and a direct-cost multiplier that are less severe than finance's case but not as
   optimistic as the original growth plan (revenue 1.10, direct cost 1.08). Have a business reason for your choice.
3. **Before running,** discuss with your neighbour: compared with original growth, will each case raise or lower the
   operating result, and which line will matter most?

If you want help translating percentages into multipliers, ask Copilot:

> Finance expects sales about 5% above a baseline and supplier costs about 12% above the baseline. Costs are stored as
> negative numbers. Explain how to turn each percentage into a multiplier applied to the baseline amount, and what a
> multiplier above 1 does to a negative cost. Do not choose my own scenario for me.

**Do:** change the four numbers, then run the cell. **You should see:** your two cases in a small table and
`Saved outputs/D2-S05/changed_assumptions.csv`.

In [ ]:
# YOUR CHOICE: write each multiplier as a number, like 1.05. Overhead stays at 1.00.
FINANCE_REVENUE_MULTIPLIER = 1.00
FINANCE_DIRECT_COST_MULTIPLIER = 1.00
MY_REVENUE_MULTIPLIER = 1.00
MY_DIRECT_COST_MULTIPLIER = 1.00

# SUPPLIED: build and save the two cases.
chosen = planning.chosen_scenarios(baseline, FINANCE_REVENUE_MULTIPLIER, FINANCE_DIRECT_COST_MULTIPLIER,
                                   MY_REVENUE_MULTIPLIER, MY_DIRECT_COST_MULTIPLIER)

## Step 6: Check one month by hand (10 minutes)
**Why:** the table and the chart come from the same code. A calculator is an independent check.

**Do:** run the cell. It shows N02's February 2026 lines under each case. For **your** case, multiply each baseline
line by your multiplier with a calculator, round each to cents, and add the three lines. Does it match the
operating result in the table? Was your prediction from Step 5 right?

In [ ]:
# SUPPLIED: N02, February 2026, under original growth, finance's case and your case.
n02_february = planning.compare_n02_february(baseline, original, chosen)

### Check your result
After your own calculation. With finance's case set to revenue **1.05** and direct cost **1.12**:

| N02 February 2026 | Original growth EUR | Finance case EUR |
|---|---:|---:|
| Revenue | 57,957.77 | 55,323.32 |
| Direct cost | -21,220.86 | -22,006.81 |
| Overhead | -7,650.54 | -7,650.54 |
| Operating result | 29,086.37 | 25,665.97 |

Finance's case is **EUR 3,420.40 below** original growth: revenue contributes -2,634.45 and direct cost -785.95.
If your finance case differs, check how you translated the percentages. Your own case has its own numbers.

## Step 7: Look at the cases over time (10 minutes)
**Why:** the CFO will look at the picture first. It must separate what happened from what is assumed.

**Do:** run the cell. Check that you can tell observed history from the three future cases without relying on colour,
and that the December 2025 cutoff is visible.

**Discuss:** in Step 3, this method's revenue forecasts have missed by about EUR 2,750 per unit and month. Finance's
case changes N02's February revenue by about EUR 2,630 compared with original growth. These are different kinds of
number, one a past error and one a difference between assumptions, but they are the same size. What does that tell
the CFO about how much weight to put on the difference between the cases?

In [ ]:
# SUPPLIED: draw and save the scenario chart.
planning.scenario_chart(history, original, chosen)

## Step 8: Recommend (15 minutes)
**Do:** in the Explorer, right-click `outputs/D2-S05`, choose **New File**, name it `interpretation.md`, and write a
short paragraph under each heading:

```text
Scope (units, months, currency):
Finance's case and my case, with my reason:
What I expected, and what the hand check showed:
The biggest driver of the difference:
My recommendation to the CFO:
Limits: what the baseline may miss, and why the cases have no probabilities:
```

Then choose **Restart** in the notebook toolbar and **Run All**. The same numbers should appear: your result can be
reproduced.

**Hand in:** `outputs/D2-S05/changed_assumptions.csv`, `scenario_figure.png` and `interpretation.md`.

**Optional extension:** with the instructor, discuss one alternative to seasonal naive (for example last year's same
month adjusted by the recent trend). How would you test it on the same holdout without looking at 2026?